# SkyRecon — VisDrone Fine-Tuning on Colab GPU
**Estimated time: 2-3 hours on free T4 GPU**

### Steps:
1. Install dependencies
2. Download VisDrone dataset directly (no upload needed)
3. Convert to YOLO format
4. Train yolov8x on VisDrone
5. Download best.pt to your PC

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Step 2: Install dependencies ──────────────────────────────
!pip install ultralytics>=8.3.0 -q

In [ ]:
# ── Step 3: Download VisDrone dataset directly ─────────────────
from ultralytics.utils.downloads import download
from pathlib import Path

SAVE_DIR = Path('visdrone_dataset')
SAVE_DIR.mkdir(exist_ok=True)

URLS = [
    'https://ultralytics.com/assets/VisDrone2019-DET-train.zip',
    'https://ultralytics.com/assets/VisDrone2019-DET-val.zip',
]

print('Downloading VisDrone dataset (~1.5GB)...')
for url in URLS:
    print(f'→ {url.split("/")[-1]}')
    download(url, dir=SAVE_DIR, unzip=True, delete=True, threads=4)

print('Download complete!')

In [ ]:
# ── Step 4: Convert VisDrone → YOLO format ─────────────────────
import os
import shutil
import cv2
from pathlib import Path

VISDRONE_DIR = 'visdrone_dataset'
YOLO_DIR     = 'visdrone_yolo'

VISDRONE_TO_YOLO = {
    1: 0,   # pedestrian → person
    2: 0,   # people     → person
    3: 1,   # bicycle    → bicycle
    4: 2,   # car        → car
    5: 2,   # van        → car
    6: 7,   # truck      → truck
    7: 5,   # tricycle   → motorcycle
    8: 5,   # awning-tricycle → motorcycle
    9: 5,   # bus        → bus
    10: 3,  # motor      → motorcycle
}

YOLO_CLASSES = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']

SPLITS = {
    'train': f'{VISDRONE_DIR}/VisDrone2019-DET-train',
    'val':   f'{VISDRONE_DIR}/VisDrone2019-DET-val',
}

def convert_annotation(ann_path, img_w, img_h):
    lines = []
    for raw in ann_path.read_text().strip().splitlines():
        parts = raw.strip().split(',')
        if len(parts) < 6:
            continue
        x, y, w, h = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
        cat = int(parts[5])
        if cat == 0 or cat not in VISDRONE_TO_YOLO:
            continue
        if w <= 0 or h <= 0:
            continue
        yolo_cls = VISDRONE_TO_YOLO[cat]
        cx = (x + w / 2) / img_w
        cy = (y + h / 2) / img_h
        nw = w / img_w
        nh = h / img_h
        lines.append(f'{yolo_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
    return lines

def convert_split(split_name, split_dir):
    img_dir = Path(split_dir) / 'images'
    ann_dir = Path(split_dir) / 'annotations'
    out_img = Path(YOLO_DIR) / split_name / 'images'
    out_lbl = Path(YOLO_DIR) / split_name / 'labels'
    out_img.mkdir(parents=True, exist_ok=True)
    out_lbl.mkdir(parents=True, exist_ok=True)
    images = sorted(img_dir.glob('*.jpg')) + sorted(img_dir.glob('*.png'))
    print(f'Converting {split_name}: {len(images)} images...')
    converted = 0
    for img_path in images:
        ann_path = ann_dir / (img_path.stem + '.txt')
        if not ann_path.exists():
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]
        yolo_lines = convert_annotation(ann_path, img_w, img_h)
        if not yolo_lines:
            continue
        shutil.copy2(img_path, out_img / img_path.name)
        (out_lbl / (img_path.stem + '.txt')).write_text('\n'.join(yolo_lines))
        converted += 1
    print(f'  Done: {converted} images converted')

for split_name, split_dir in SPLITS.items():
    convert_split(split_name, split_dir)

# Write dataset YAML
yaml_content = f"""path: /content/visdrone_yolo
train: train/images
val:   val/images
nc: {len(YOLO_CLASSES)}
names: {YOLO_CLASSES}
"""
Path(f'{YOLO_DIR}/visdrone.yaml').write_text(yaml_content)
print('Dataset YAML written!')
print('Ready to train!')

In [ ]:
# ── Step 5: Train ──────────────────────────────────────────────
# Expected time: ~2-3 hours on free T4 GPU
from ultralytics import YOLO

model = YOLO('yolov8x.pt')  # downloads automatically ~130MB

results = model.train(
    data='/content/visdrone_yolo/visdrone.yaml',
    epochs=50,
    imgsz=640,
    batch=16,          # GPU can handle 16
    workers=4,
    device=0,          # GPU
    project='runs/detect',
    name='skyrecon_visdrone',
    exist_ok=True,
    patience=10,
    save=True,
    save_period=10,
    val=True,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    freeze=10,
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    cos_lr=True,
    label_smoothing=0.1,
    verbose=True,
)

print('Training complete!')
print(f'Best model: runs/detect/skyrecon_visdrone/weights/best.pt')

In [ ]:
# ── Step 6: Download best.pt to your PC ───────────────────────
from google.colab import files
files.download('runs/detect/skyrecon_visdrone/weights/best.pt')
print('Download started! Save it to SkyRecon/backend/ on your PC')

## After downloading best.pt

1. Rename it to `skyrecon_visdrone.pt`
2. Copy it to `SkyRecon/backend/`
3. Open `SkyRecon/backend/.env` and change:
```
YOLO_MODEL=skyrecon_visdrone.pt
```
4. Restart the backend — done! 🚀